In [20]:
!pip install gradio scikit-image --quiet
print("Done Installing Libraries")

Done Installing Libraries


In [21]:
#Importing Everything need for this Project

import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader          
from torch.cuda.amp import GradScaler, autocast  

import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

# ---- Check GPU ----
print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Number of GPUs  : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i} → {torch.cuda.get_device_name(i)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing: {device}")

def fix_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fix_seed(42)
print("Ready to go")

PyTorch version : 2.9.0+cu126
GPU available   : True
Number of GPUs  : 2
  GPU 0 → Tesla T4
  GPU 1 → Tesla T4

Using: cuda
Ready to go


In [22]:
#All settings in one place
class Config:
    #Image Setup
    IMAGE_SIZE = 224
    PATCH_SIZE = 16
    NUM_PATCHES = (224 // 16) ** 2 #=196 PATCHES
    MASK_RATIO = 0.75
    #ENCODER
    ENC_DIM = 768
    ENC_LAYERS = 12
    ENC_HEADS = 12
    ENC_MLP_RATIO = 4
    #DECODER
    DEC_DIM = 384
    DEC_LAYERS = 12
    DEC_HEADS = 6
    DEC_MLP_RATIO = 4
    #TRAINING
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LR = 1.5e-4
    WEIGHT_DECAY= 0.05
    WARMUP_EPOCHS = 5
    GRAD_CLIP = 1.0
    #PATHS
    DATA_DIR      = "/kaggle/input/datasets/akash2sharma/tiny-imagenet/tiny-imagenet-200/tiny-imagenet-200"
    TRAIN_DIR     = DATA_DIR + "/train"
    VAL_DIR       = DATA_DIR + "/val"
    SAVE_DIR      = "/kaggle/working/checkpoints"

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print("Checking paths...")
print(f"  DATA_DIR  exists: {os.path.exists(cfg.DATA_DIR)}")
print(f"  TRAIN_DIR exists: {os.path.exists(cfg.TRAIN_DIR)}")
print(f"  VAL_DIR   exists: {os.path.exists(cfg.VAL_DIR)}")
print(f"\nTotal patches   : {cfg.NUM_PATCHES}")
print(f"Visible patches : {int(cfg.NUM_PATCHES * (1 - cfg.MASK_RATIO))}  (25%)")
print(f"Masked patches  : {int(cfg.NUM_PATCHES * cfg.MASK_RATIO)}  (75%)")
print("Config set!!")

Checking paths...
  DATA_DIR  exists: True
  TRAIN_DIR exists: True
  VAL_DIR   exists: True

Total patches   : 196
Visible patches : 49  (25%)
Masked patches  : 147  (75%)
Config set!!


In [23]:
import shutil

WORK_VAL_DIR = "/kaggle/working/val_fixed"

def fix_tiny_imagenet_val(src_val_dir, dst_val_dir):
    """
    Kaggle /kaggle/input is READ-ONLY.
    This copies val images into /kaggle/working/val_fixed/
    organized by class name so ImageFolder can read it.

    From:  src_val_dir/images/val_0.JPEG
    To:    dst_val_dir/class_name/val_0.JPEG
    """
    if os.path.exists(dst_val_dir) and len(os.listdir(dst_val_dir)) > 0:
        print(f"Val folder already fixed at: {dst_val_dir}")
        return

    os.makedirs(dst_val_dir, exist_ok=True)

    anno_file  = os.path.join(src_val_dir, "val_annotations.txt")
    images_dir = os.path.join(src_val_dir, "images")

    if not os.path.exists(anno_file):
        print("val_annotations.txt not found!")
        return

    img_to_class = {}
    with open(anno_file, "r") as f:
        for line in f.readlines():
            parts = line.strip().split("\t")
            img_to_class[parts[0]] = parts[1]

    print(f"Found {len(img_to_class):,} val images across "
          f"{len(set(img_to_class.values()))} classes")
    print("Copying val images to working directory...")

    for img_name, class_name in img_to_class.items():
        class_dir = os.path.join(dst_val_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        src = os.path.join(images_dir, img_name)
        dst = os.path.join(class_dir, img_name)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)

    print(f"Val images organized at: {dst_val_dir}")


# ---- Run the fix ----
fix_tiny_imagenet_val(
    src_val_dir = cfg.VAL_DIR,      
    dst_val_dir = WORK_VAL_DIR      
)

# ---- Image transforms ----
train_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ---- Load datasets ----
train_dataset = ImageFolder(
    root      = cfg.TRAIN_DIR,      
    transform = train_transform
)

val_dataset = ImageFolder(
    root      = WORK_VAL_DIR,       
)

train_loader = DataLoader(
    train_dataset,
    batch_size  = cfg.BATCH_SIZE,
    shuffle     = True,
    num_workers = 4,
    pin_memory  = True
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = cfg.BATCH_SIZE,
    shuffle     = False,
    num_workers = 4,
    pin_memory  = True
)

print(f"\nTraining images : {len(train_dataset):,}")
print(f"Validation imgs : {len(val_dataset):,}")
print(f"Train batches   : {len(train_loader):,}")
print(f"Val batches     : {len(val_loader):,}")
print("Dataset loaded successfully!")





Val folder already fixed at: /kaggle/working/val_fixed

Training images : 100,000
Validation imgs : 10,000
Train batches   : 3,125
Val batches     : 313
Dataset loaded successfully!


In [24]:
class PatchEmbed(nn.Module):
    def __init__(self, image_size=224, patch_size=16,
                 in_channels=3, embed_dim=768):

        super().__init__()

        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size

        # patching + projection
        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):

        x = self.proj(x)      # (B, embed_dim, H/P, W/P)
        x = x.flatten(2)      # (B, embed_dim, N)
        x = x.transpose(1, 2) # (B, N, embed_dim)

        return x


def random_masking(x, mask_ratio=0.75):

    # randomly keep 25% patches
    B, N, D = x.shape
    num_keep = int(N * (1 - mask_ratio))

    noise = torch.rand(B, N, device=x.device)

    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    ids_keep = ids_shuffle[:, :num_keep]

    x_visible = torch.gather(
        x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D)
    )

    mask = torch.ones(B, N, device=x.device)
    mask[:, :num_keep] = 0
    mask = torch.gather(mask, 1, ids_restore)

    return x_visible, mask, ids_restore


print("PatchEmbed and masking is ready")

PatchEmbed and masking is ready


In [25]:
def get_2d_sincos_pos_embed(embed_dim, grid_size):
    
    # Create grid
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)

    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0)

    grid = grid.reshape(2, 1, grid_size, grid_size)

    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)

    return pos_embed


def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):

    assert embed_dim % 2 == 0

    emb_h = get_1d_sincos_pos_embed(embed_dim // 2, grid[0].reshape(-1))
    emb_w = get_1d_sincos_pos_embed(embed_dim // 2, grid[1].reshape(-1))

    emb = np.concatenate([emb_h, emb_w], axis=1)

    return emb


def get_1d_sincos_pos_embed(embed_dim, pos):

    omega = np.arange(embed_dim // 2, dtype=np.float32)
    omega = 1. / (10000 ** (omega / (embed_dim / 2)))

    pos = pos.reshape(-1)

    out = np.outer(pos, omega)

    emb_sin = np.sin(out)
    emb_cos = np.cos(out)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)

    return emb


print("Positional Embedding ready!")

Positional Embedding ready!


In [26]:
# Transformer Building Blocks

class MultiheadSelfAttention(nn.Module):

    # Each patch attends to all other patches
    def __init__(self, dim, num_heads, dropout=0.0):

        super().__init__()

        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, N, C = x.shape

        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)

        x = self.proj(x)

        return x


class FeedForward(nn.Module):

    # MLP inside transformer
    def __init__(self, dim, mlp_ratio=4, dropout=0.0):

        super().__init__()

        hidden = int(dim * mlp_ratio)

        self.net = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):

        return self.net(x)


class TransformerBlock(nn.Module):

    # Attention + FeedForward block
    def __init__(self, dim, num_heads, mlp_ratio=4, dropout=0.0):

        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = MultiheadSelfAttention(dim, num_heads, dropout)

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, mlp_ratio, dropout)

    def forward(self, x):

        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))

        return x


print("Transformer blocks ready!")

Transformer blocks ready!


In [27]:
class MAEEncoder(nn.Module):

    # large ViT encoder only sees 49 patches (25%)
    def __init__(self, image_size=224, patch_size=16,
                 in_channels=3, embed_dim=768,
                 depth=12, num_heads=12, mlp_ratio=4):

        super().__init__()

        self.embed_dim = embed_dim

        num_patches = (image_size // patch_size) ** 2

        self.patch_embed = PatchEmbed(
            image_size, patch_size, in_channels, embed_dim
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, embed_dim),
            requires_grad=False
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)

        self._init_weights()

    def _init_weights(self):

        num_patches = self.pos_embed.shape[1] - 1
        grid_size = int(num_patches ** 0.5)

        pos_embed = get_2d_sincos_pos_embed(self.embed_dim, grid_size)

        self.pos_embed.data[0, 1:, :] = torch.from_numpy(pos_embed)

        nn.init.normal_(self.cls_token, std=0.02)

        self.apply(self._init_module)

    def _init_module(self, m):

        if isinstance(m, nn.Linear):

            nn.init.xavier_uniform_(m.weight)

            if m.bias is not None:
                nn.init.zeros_(m.bias)

        elif isinstance(m, nn.LayerNorm):

            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, imgs, mask_ratio=0.75):

        # Patchify image
        x = self.patch_embed(imgs)

        # Add positional embedding
        x = x + self.pos_embed[:, 1:, :]

        # Mask 75% patches
        x, mask, ids_restore = random_masking(x, mask_ratio)

        # Add CLS token
        cls_token = self.cls_token + self.pos_embed[:, :1, :]
        cls_tokens = cls_token.expand(x.shape[0], -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        return x, mask, ids_restore


print("MAE Encoder ready!")

MAE Encoder ready!


In [28]:
class MAEDecoder(nn.Module):

    # Reconstructs all masked patches
    def __init__(self,
                 num_patches=196,
                 patch_size=16,
                 encoder_dim=768,
                 decoder_dim=384,
                 depth=12,
                 num_heads=6,
                 mlp_ratio=4):

        super().__init__()

        self.num_patches = num_patches
        self.patch_size = patch_size

        # encoder dim → decoder dim
        self.encoder_to_decoder = nn.Linear(encoder_dim, decoder_dim)

        # learnable mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))

        self.pos_embed = nn.Parameter(
            torch.zeros(1, num_patches + 1, decoder_dim),
            requires_grad=False
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(decoder_dim, num_heads, mlp_ratio)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(decoder_dim)

        # prediction head
        self.pred_head = nn.Linear(
            decoder_dim,
            patch_size * patch_size * 3
        )

        self._init_weights()

    def _init_weights(self):

        grid_size = int(self.num_patches ** 0.5)

        pos_embed = get_2d_sincos_pos_embed(
            self.pos_embed.shape[-1],
            grid_size
        )

        self.pos_embed.data[0, 1:, :] = torch.from_numpy(pos_embed)

        nn.init.normal_(self.mask_token, std=0.02)

        self.apply(self._init_module)

    def _init_module(self, m):

        if isinstance(m, nn.Linear):

            nn.init.xavier_uniform_(m.weight)

            if m.bias is not None:
                nn.init.zeros_(m.bias)

        elif isinstance(m, nn.LayerNorm):

            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, encoder_out, ids_restore):

        # 1️⃣ encoder → decoder dim
        x = self.encoder_to_decoder(encoder_out)

        # separate cls token
        cls_token = x[:, :1, :]
        x = x[:, 1:, :]

        # 2️⃣ create mask tokens
        num_mask = ids_restore.shape[1] - x.shape[1]

        mask_tokens = self.mask_token.repeat(
            x.shape[0],
            num_mask,
            1
        )

        # 3️⃣ visible + mask tokens
        x_ = torch.cat([x, mask_tokens], dim=1)

        # 4️⃣ restore original order
        x_ = torch.gather(
            x_,
            1,
            ids_restore.unsqueeze(-1).expand(-1, -1, x_.shape[2])
        )

        # add cls token back
        x = torch.cat([cls_token, x_], dim=1)

        # add positional embedding
        x = x + self.pos_embed

        # 5️⃣ transformer blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # 6️⃣ predict pixels (remove cls)
        x = self.pred_head(x[:, 1:, :])

        return x


print("MAE Decoder ready!")

MAE Decoder ready!


In [29]:
class MAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.encoder = MAEEncoder(
            image_size = cfg.IMAGE_SIZE,
            patch_size = cfg.PATCH_SIZE,
            embed_dim  = cfg.ENC_DIM,
            depth      = cfg.ENC_LAYERS,
            num_heads  = cfg.ENC_HEADS,
            mlp_ratio  = cfg.ENC_MLP_RATIO,
        )
        self.decoder = MAEDecoder(
            num_patches  = cfg.NUM_PATCHES,
            patch_size   = cfg.PATCH_SIZE,
            encoder_dim  = cfg.ENC_DIM,
            decoder_dim  = cfg.DEC_DIM,
            depth        = cfg.DEC_LAYERS,
            num_heads    = cfg.DEC_HEADS,
            mlp_ratio    = cfg.DEC_MLP_RATIO,
        )
        self.patch_size = cfg.PATCH_SIZE

    def patchify(self, imgs):
        """Image → flat patches   (B,3,224,224) → (B,196,768)"""
        p = self.patch_size
        B, C, H, W = imgs.shape
        h = H // p
        w = W // p
        x = imgs.reshape(B, C, h, p, w, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(B, h * w, C * p * p)
        return x

    def unpatchify(self, patches):
        """Flat patches → image   (B,196,768) → (B,3,224,224)"""
        p = self.patch_size
        B, N, _ = patches.shape
        h = w = int(N ** 0.5)
        x = patches.reshape(B, h, w, 3, p, p)
        x = x.permute(0, 3, 1, 4, 2, 5)
        x = x.reshape(B, 3, h * p, w * p)
        return x

    def forward(self, imgs, mask_ratio=0.75):
        # Encoder
        latent, mask, ids_restore = self.encoder(imgs, mask_ratio)
        # Decoder
        pred   = self.decoder(latent, ids_restore)      # (B, 196, 768)
        # Target: normalize pixel values per patch
        target = self.patchify(imgs)
        mean   = target.mean(dim=-1, keepdim=True)
        var    = target.var(dim=-1, keepdim=True)
        target = (target - mean) / (var + 1e-6).sqrt()
        # MSE loss ONLY on masked patches
        loss   = (pred - target) ** 2
        loss   = loss.mean(dim=-1)                      # (B, 196)
        loss   = (loss * mask).sum() / mask.sum()
        return loss, pred, mask


# Build model + wrap in DataParallel for dual GPU
model = MAE(cfg).to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total / 1e6:.1f} Million")
print("Full MAE model ready!")

Using 2 GPUs!
Total parameters: 107.8 Million
Full MAE model ready!


In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = cfg.LR,
    weight_decay = cfg.WEIGHT_DECAY,
    betas        = (0.9, 0.95),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max   = cfg.NUM_EPOCHS,
    eta_min = cfg.LR * 0.01,
)

scaler = GradScaler()

print("Optimizer, Scheduler, Scaler ready!")

Optimizer, Scheduler, Scaler ready!


/tmp/ipykernel_55/159059273.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# ---- Main training loop ----
scaler = torch.cuda.amp.GradScaler()  # fresh scaler

def warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    if epoch < warmup_epochs:
        lr = base_lr * (epoch + 1) / warmup_epochs
        for pg in optimizer.param_groups:
            pg["lr"] = lr

def train_one_epoch(model, loader, optimizer, scaler, device, epoch, cfg):
    model.train()
    total_loss = 0.0
    num_batches = 0
    for batch_idx, (imgs, _) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        with autocast():
            loss, pred, mask = model(imgs, mask_ratio=cfg.MASK_RATIO)
            if isinstance(loss, torch.Tensor) and loss.ndim > 0:
                loss = loss.mean()
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        total_loss = total_loss + loss.item()
        num_batches = num_batches + 1
        if (batch_idx + 1) % 100 == 0:
            avg = total_loss / num_batches
            lr = optimizer.param_groups[0]["lr"]
            print(f" Epoch {epoch+1}| Batch {batch_idx+1}/{len(loader)}"
                  f"|loss:{avg:.4f}|LR:{lr:.6f}")
    return total_loss / num_batches

train_losses = []
best_loss = float("inf")
print("Starting training!")
print("=" * 60)

for epoch in range(cfg.NUM_EPOCHS):
    warmup_lr(optimizer, epoch, cfg.WARMUP_EPOCHS, cfg.LR)
    avg_loss = train_one_epoch(
        model, train_loader, optimizer, scaler, device, epoch, cfg
    )
    if epoch >= cfg.WARMUP_EPOCHS:
        scheduler.step()
    train_losses.append(avg_loss)
    print(f"\nEpoch {epoch+1}/{cfg.NUM_EPOCHS} | "
          f"Train Loss: {avg_loss:.4f} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    if avg_loss < best_loss:
        best_loss = avg_loss
        save_path = os.path.join(cfg.SAVE_DIR, "mae_best.pth")
        state = (model.module.state_dict()
                 if hasattr(model, "module") else model.state_dict())
        torch.save(state, save_path)
        print(f" New best model saved! Loss: {best_loss:.4f}")
    print("-" * 60)

print("\n Training Complete!")
print(f"Best training loss: {best_loss:.4f}")

Starting training!


/tmp/ipykernel_55/963699746.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # fresh scaler
/tmp/ipykernel_55/963699746.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


 Epoch 1| Batch 100/3125|loss:0.9651|LR:0.000030
 Epoch 1| Batch 200/3125|loss:0.8864|LR:0.000030
 Epoch 1| Batch 300/3125|loss:0.8531|LR:0.000030
 Epoch 1| Batch 400/3125|loss:0.8351|LR:0.000030
 Epoch 1| Batch 500/3125|loss:0.8215|LR:0.000030
 Epoch 1| Batch 600/3125|loss:0.8073|LR:0.000030
 Epoch 1| Batch 700/3125|loss:0.7944|LR:0.000030
 Epoch 1| Batch 800/3125|loss:0.7825|LR:0.000030
 Epoch 1| Batch 900/3125|loss:0.7707|LR:0.000030
 Epoch 1| Batch 1000/3125|loss:0.7606|LR:0.000030
 Epoch 1| Batch 1100/3125|loss:0.7509|LR:0.000030
 Epoch 1| Batch 1200/3125|loss:0.7432|LR:0.000030
 Epoch 1| Batch 1300/3125|loss:0.7367|LR:0.000030
 Epoch 1| Batch 1400/3125|loss:0.7296|LR:0.000030
 Epoch 1| Batch 1500/3125|loss:0.7239|LR:0.000030
 Epoch 1| Batch 1600/3125|loss:0.7187|LR:0.000030
 Epoch 1| Batch 1700/3125|loss:0.7138|LR:0.000030
 Epoch 1| Batch 1800/3125|loss:0.7096|LR:0.000030
 Epoch 1| Batch 1900/3125|loss:0.7053|LR:0.000030
 Epoch 1| Batch 2000/3125|loss:0.7010|LR:0.000030
 Epoch 1|

In [ ]:
# PLOT TRAINING LOSS CURVE


plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses,
         color="royalblue", linewidth=2, marker="o", markersize=4)
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("Reconstruction Loss (MSE)", fontsize=13)
plt.title("MAE Training Loss vs Epochs", fontsize=15)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/training_loss.png", dpi=150)
plt.show()
print("Loss curve saved!")

In [ ]:

# VISUALIZATION — Show 5 examples side by side


IMG_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMG_STD  = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):
    t = tensor.clone().cpu()
    for c in range(3):
        t[c] = t[c] * IMG_STD[c] + IMG_MEAN[c]
    return t.clamp(0, 1)

def make_masked_image(img, mask, patch_size=16):
    img_show = denormalize(img).permute(1, 2, 0).numpy()
    mask_img = img_show.copy()
    H = W = img_show.shape[0]
    num_per_row = W // patch_size
    for i in range(mask.shape[0]):
        if mask[i] == 1:
            row = (i // num_per_row) * patch_size
            col = (i  % num_per_row) * patch_size
            mask_img[row:row+patch_size, col:col+patch_size] = 0.5
    return mask_img

def visualize_reconstruction(model, loader, device, cfg, num_samples=5):
    raw_model = model.module if hasattr(model, "module") else model
    raw_model.eval()

    imgs, _ = next(iter(loader))
    imgs = imgs[:num_samples].to(device)

    with torch.no_grad():
        with autocast():
            loss, pred, mask = raw_model(imgs, mask_ratio=cfg.MASK_RATIO)

    recon_imgs = raw_model.unpatchify(pred).cpu()

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 4))
    titles = ["Masked Input (75% hidden)", "Reconstruction", "Original"]

    for i in range(num_samples):
        masked_view = make_masked_image(imgs[i].cpu(), mask[i].cpu(), cfg.PATCH_SIZE)
        recon_view  = denormalize(recon_imgs[i]).permute(1, 2, 0).numpy()
        orig_view   = denormalize(imgs[i].cpu()).permute(1, 2, 0).numpy()

        for j, (img_data, title) in enumerate(zip(
            [masked_view, recon_view, orig_view], titles
        )):
            axes[i][j].imshow(img_data.clip(0, 1))
            axes[i][j].set_title(title, fontsize=11, fontweight="bold")
            axes[i][j].axis("off")

    plt.suptitle("MAE Reconstruction Results", fontsize=16, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("/kaggle/working/reconstruction_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Visualization saved!")

    return imgs.cpu(), recon_imgs, mask.cpu()


orig_imgs, recon_imgs, masks = visualize_reconstruction(
    model, val_loader, device, cfg, num_samples=5
)

In [ ]:

# QUANTITATIVE EVALUATION — PSNR and SSIM


from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

def compute_psnr_ssim(original_batch, recon_batch):
    psnr_scores = []
    ssim_scores = []

    for i in range(original_batch.shape[0]):
        orig  = denormalize(original_batch[i]).permute(1, 2, 0).numpy()
        recon = denormalize(recon_batch[i]).permute(1, 2, 0).numpy()

        orig  = orig.clip(0, 1).astype(np.float32)
        recon = recon.clip(0, 1).astype(np.float32)

        psnr  = calc_psnr(orig, recon, data_range=1.0)
        ssim  = calc_ssim(orig, recon, data_range=1.0,
                          channel_axis=2,
                          win_size=7)

        psnr_scores.append(psnr)
        ssim_scores.append(ssim)

    return np.mean(psnr_scores), np.mean(ssim_scores), psnr_scores, ssim_scores


avg_psnr, avg_ssim, all_psnr, all_ssim = compute_psnr_ssim(orig_imgs, recon_imgs)

print("=" * 40)
print("Quantitative Evaluation Results")
print("=" * 40)
print(f"Average PSNR : {avg_psnr:.2f} dB")
print(f"Average SSIM : {avg_ssim:.4f}")
print("-" * 40)
for i, (p, s) in enumerate(zip(all_psnr, all_ssim)):
    print(f"  Sample {i+1}  →  PSNR: {p:.2f} dB  |  SSIM: {s:.4f}")
print("=" * 40)

In [ ]:

# GRADIO APP


import gradio as gr

# Load best model
inference_model = MAE(cfg).to(device)
ckpt_path = os.path.join(cfg.SAVE_DIR, "mae_best.pth")

if os.path.exists(ckpt_path):
    inference_model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print("Best checkpoint loaded for inference!")
else:
    print("No checkpoint found — using current model weights.")
    raw_model = model.module if hasattr(model, "module") else model
    inference_model.load_state_dict(raw_model.state_dict())

inference_model.eval()

app_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def run_mae_app(input_image, mask_ratio_pct):
    if input_image is None:
        return None, None, None, "Please upload an image first!"

    mask_ratio = float(mask_ratio_pct) / 100.0
    img_tensor = app_transform(input_image).unsqueeze(0).to(device)

    with torch.no_grad():
        with autocast():
            loss, pred, mask = inference_model(img_tensor, mask_ratio=mask_ratio)

    recon_img = inference_model.unpatchify(pred)[0].cpu()

    masked_np = make_masked_image(img_tensor[0].cpu(), mask[0].cpu(), cfg.PATCH_SIZE)
    recon_np  = denormalize(recon_img).permute(1, 2, 0).numpy().clip(0, 1)
    orig_np   = denormalize(img_tensor[0].cpu()).permute(1, 2, 0).numpy().clip(0, 1)

    psnr_val = calc_psnr(orig_np, recon_np, data_range=1.0)
    ssim_val = calc_ssim(orig_np, recon_np, data_range=1.0,
                         channel_axis=2, win_size=7)

    info = (f"Reconstruction Done!\n"
            f"Mask Ratio : {mask_ratio_pct}%\n"
            f"PSNR       : {psnr_val:.2f} dB\n"
            f"SSIM       : {ssim_val:.4f}\n"
            f"Loss       : {loss.item():.4f}")

    to_uint8 = lambda arr: (arr * 255).astype(np.uint8)
    return (to_uint8(masked_np),
            to_uint8(recon_np),
            to_uint8(orig_np),
            info)


with gr.Blocks(title="MAE Image Reconstruction", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # Masked Autoencoder (MAE) — Image Reconstruction Demo
    Upload any image and watch the model reconstruct it from only a fraction of visible patches!
    """)

    with gr.Row():
        with gr.Column(scale=1):
            input_img   = gr.Image(type="pil", label="Upload Image")
            mask_slider = gr.Slider(minimum=10, maximum=95, value=75, step=5,
                                    label="Masking Ratio (%)")
            run_btn     = gr.Button("Run Reconstruction", variant="primary")

        with gr.Column(scale=2):
            with gr.Row():
                masked_out = gr.Image(label="Masked Input")
                recon_out  = gr.Image(label="Reconstruction")
                orig_out   = gr.Image(label="Original")

    info_box = gr.Textbox(label="Metrics", lines=5)

    run_btn.click(
        fn      = run_mae_app,
        inputs  = [input_img, mask_slider],
        outputs = [masked_out, recon_out, orig_out, info_box]
    )

    gr.Markdown("""
    ### How it works:
    1. Image is divided into **196 patches** (14×14 grid)
    2. Selected % of patches are **randomly hidden**
    3. MAE encoder reads **only visible patches**
    4. Decoder reconstructs the **full image** from partial information
    """)

demo.launch(share=True, debug=False)